# Proyecto: Random Forest con dataset existente



Flujo completo en este notebook:



1. Cargar una base de datos existente para entrenamiento.

2. Entrenar RandomForest una sola vez.

3. Guardar el modelo aprendido en disco.

4. Cargar el modelo guardado y usarlo solo para predecir.



No se usan endpoints: toda la prueba es local en este notebook.


In [ ]:
%pip install scikit-learn joblib pandas datasets

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.9 MB 10.5 MB/s eta 0:00:01
   --------------------- ------------------ 5.2/9.9 MB 15.2 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.9 MB 18.0 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 17.1 MB/s  0:00:00

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas

In [1]:
from datetime import datetime
from pathlib import Path
import os
import random

import joblib
import pandas as pd
from datasets import load_dataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

c:\Users\ders1\OneDrive\Documentos\gthub\Proyecto-PTIA-Back\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Cargar base de datos existente



Se usa el dataset publico `tweet_eval` en su configuracion `sentiment`.



Etiquetas originales:

- 0: negative

- 1: neutral

- 2: positive



Mapeo al proyecto:

- negative -> toxico

- neutral -> moderado

- positive -> seguro

In [2]:
dataset = load_dataset('tweet_eval', 'sentiment')

label_map = {0: 'toxico', 1: 'moderado', 2: 'seguro'}

df_train = pd.DataFrame(dataset['train'])
df_valid = pd.DataFrame(dataset['validation'])
df_test = pd.DataFrame(dataset['test'])

df = pd.concat([df_train, df_valid, df_test], ignore_index=True)
df = df.rename(columns={'text': 'text_raw', 'label': 'label_raw'})
df['label'] = df['label_raw'].map(label_map)
df = df[['text_raw', 'label']].rename(columns={'text_raw': 'text'})

print('Total filas:', len(df))
print('Distribucion de clases:')
print(df['label'].value_counts())
df.head()

c:\Users\ders1\OneDrive\Documentos\gthub\Proyecto-PTIA-Back\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ders1\.cache\huggingface\hub\datasets--tweet_eval. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 2000/2000 [00:00<00:00, 642903.74 examples/s]

Total filas: 59899
Distribucion de clases:
label
moderado    27479
seguro      21043
toxico      11377
Name: count, dtype: int64


,text,label
0,"""QT @user In the original draft of the 7th boo...",seguro
1,"""Ben Smith / Smith (concussion) remains out of...",moderado
2,Sorry bout the stream last night I crashed out...,moderado
3,Chase Headley's RBI double in the 8th inning o...,moderado
4,@user Alciato: Bee will invest 150 million in ...,seguro


## 2) Split de entrenamiento y prueba

In [3]:
X = df['text'].astype(str).tolist()
y = df['label'].astype(str).tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Train:', len(X_train), 'Test:', len(X_test))

Train: 47919 Test: 11980


## 3) Entrenamiento una sola vez con Random Forest

In [4]:
pipeline = Pipeline(
    steps=[
        ('vectorizer', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)),
        ('classifier', RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight='balanced_subsample',
            n_jobs=-1
        ))
    ]
)

pipeline.fit(X_train, y_train)
print('Entrenamiento finalizado en', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

Entrenamiento finalizado en 2026-04-06 00:53:27


## 4) Evaluar, guardar modelo aprendido y probar inferencia



En esta etapa se guarda el entrenamiento y luego se usa solo para prediccion.

In [5]:
pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, pred)

print('Accuracy:', round(acc, 4))
print('\nClassification report:')
print(classification_report(y_test, pred, zero_division=0))

labels_order = ['toxico', 'moderado', 'seguro']
cm = confusion_matrix(y_test, pred, labels=labels_order)
print('Confusion matrix (toxico, moderado, seguro):')
print(cm)

model_path = Path('modelo_random_forest_aprendido.joblib')
bundle = {
    'pipeline': pipeline,
    'labels': list(pipeline.classes_),
    'model_type': 'RandomForestClassifier',
    'trained_on': 'tweet_eval/sentiment'
}
joblib.dump(bundle, model_path)

mtime_before = os.path.getmtime(model_path)
print('\nModelo guardado en:', model_path.resolve())
print('Timestamp inicial:', datetime.fromtimestamp(mtime_before))

Accuracy: 0.6234

Classification report:
              precision    recall  f1-score   support

    moderado       0.58      0.86      0.69      5496
      seguro       0.73      0.54      0.62      4209
      toxico       0.72      0.20      0.31      2275

    accuracy                           0.62     11980
   macro avg       0.68      0.53      0.54     11980
weighted avg       0.66      0.62      0.59     11980

Confusion matrix (toxico, moderado, seguro):
[[ 457 1593  225]
 [ 152 4727  617]
 [  25 1900 2284]]

Modelo guardado en: C:\Users\ders1\OneDrive\Documentos\gthub\Proyecto-PTIA-Back\notebooks\modelo_random_forest_aprendido.joblib
Timestamp inicial: 2026-04-06 00:53:43.663243


## 5) Simulacion de uso del modelo ya aprendido



Esta funcion simula el backend: solo carga el artefacto guardado y predice.

In [7]:
def backend_predict(text: str, saved_model_path: Path) -> dict:
    loaded = joblib.load(saved_model_path)
    model = loaded['pipeline']
    labels = loaded['labels']

    probs = model.predict_proba([text])[0]
    idx = int(probs.argmax())

    return {
        'text': text,
        'label': str(labels[idx]),
        'confidence': round(float(probs[idx]), 4),
        'source': 'random-forest'
    }

ejemplos = [
    'stupid'
 ]

print('Predicciones de prueba:')
for t in ejemplos:
    print(backend_predict(t, model_path))

for _ in range(20):
    _ = backend_predict(random.choice(ejemplos), model_path)

mtime_after = os.path.getmtime(model_path)
print('\nTimestamp final:', datetime.fromtimestamp(mtime_after))
print('Se modifico el modelo durante inferencia?:', mtime_after != mtime_before)

Predicciones de prueba:
{'text': 'stupid', 'label': 'toxico', 'confidence': 0.49, 'source': 'random-forest'}

Timestamp final: 2026-04-06 00:53:43.663243
Se modifico el modelo durante inferencia?: False


## Conclusiones



- Se uso una base de datos existente para entrenar (`tweet_eval`).

- El Random Forest se entrena una sola vez.

- El modelo aprendido queda guardado en `modelo_random_forest_aprendido.joblib`.

- Las predicciones posteriores usan solo inferencia y no reentrenan el modelo.